# 03 · Indexing and broadcasting real data / Indexación y broadcasting con datos reales

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/03-indexing-and-broadcasting.ipynb)

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#7c3aed,rgba(124,58,237,0))"></div>

<span style="font:700 11px/1.6 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#7c3aed">PART III · EXERCISE · 15 MIN</span>

This notebook teaches two practical skills:

1. **Indexing** — choosing exactly the rows, columns, or values you want.
2. **Broadcasting** — applying a smaller set of numbers across a larger array without manually copying them.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><div style="margin:0 0 .7em">Este cuaderno enseña dos habilidades prácticas:</div><div style="margin:0 0 0">1. <b>Indexación</b> — seleccionar exactamente las filas, columnas o valores que necesitas. 2. <b>Broadcasting</b> — aplicar un conjunto pequeño de números sobre un arreglo más grande sin copiarlos manualmente.</div></div>

## What you will be able to do / Lo que podrás hacer

- Select a real measurement by **name** instead of relying on a hard-coded column number.
- Use **fancy indexing** to choose several specific observations at once.
- Use a **Boolean mask** to keep only observations that satisfy a condition.
- Standardize real image data with broadcasting.
- Detect **zero-variance pixels** and explain why division by zero creates `NaN`.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><ul style="margin:0;padding-left:1.2em"><li style="margin:.35em 0">Seleccionar una medición real por <b>nombre</b> en lugar de depender de un número fijo de columna.</li><li style="margin:.35em 0">Usar <b>fancy indexing</b> para elegir varias observaciones específicas al mismo tiempo.</li><li style="margin:.35em 0">Usar una <b>máscara booleana</b> para conservar solo observaciones que cumplen una condición.</li><li style="margin:.35em 0">Estandarizar datos reales de imágenes mediante broadcasting.</li><li style="margin:.35em 0">Detectar <b>píxeles de varianza cero</b> y explicar por qué dividir entre cero produce <code>NaN</code>.</li></ul></div>

## Four ideas before we code / Cuatro ideas antes de programar

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#7c3aed,rgba(124,58,237,0))"></div>

Picture a spreadsheet.

| Concept / Concepto | Simple idea / Idea sencilla | In a spreadsheet / En una hoja de cálculo |
|---|---|---|
| **Indexing / Indexación** | Choose a position / Elegir una posición | “Give me column B” / “Dame la columna B” |
| **Fancy indexing** | Choose several at once / Elegir varias a la vez | “Give me rows 3, 8 and 20” / “Dame las filas 3, 8 y 20” |
| **Boolean mask / Máscara booleana** | Keep rows where a condition is `True` / Conservar filas donde una condición es `True` | “Only trips after 6 p.m.” / “Solo viajes después de las 6 p. m.” |
| **Broadcasting** | Reuse a small array across many rows / Reutilizar un arreglo pequeño en muchas filas | The same column rule on every row / La misma regla en cada fila |

One more word arrives later. **Standardization** subtracts a feature's mean and
divides by its standard deviation, so features become comparable.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Una palabra más para después: <b>estandarización</b> es restar la media de una característica y dividir por su desviación estándar, para que las características sean comparables.</div>

## Setup / Preparación

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#7c3aed,rgba(124,58,237,0))"></div>

Two real datasets, both packaged with scikit-learn:

- breast-cancer measurements, for the indexing exercises;
- handwritten digits, for broadcasting and zero variance.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><ul style="margin:0;padding-left:1.2em"><li style="margin:.35em 0">mediciones de cáncer de mama, para los ejercicios de indexación;</li><li style="margin:.35em 0">dígitos manuscritos, para broadcasting y varianza cero.</li></ul></div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display
from sklearn.datasets import load_breast_cancer, load_digits

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

bc = load_breast_cancer()
X = bc.data
y = bc.target              # 0 = malignant, 1 = benign
names = list(bc.feature_names)

digits = load_digits()
images = digits.images     # (1797, 8, 8)

print("Breast-cancer matrix / Matriz de cáncer de mama:", X.shape)
print("Number of feature names / Número de características:", len(names))
print("Digit images / Imágenes de dígitos:", images.shape)
print()
print("EN: Setup ready.")
print("ES: Preparación lista.")

## The mistake that does not crash / El error que no hace fallar el programa

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#7c3aed,rgba(124,58,237,0))"></div>

<div style="border-left:5px solid #d97706;background:rgba(217,119,6,.10);border-radius:0 8px 8px 0;padding:14px 18px;margin:1.8em 0;font:400 15px/1.75 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#d97706;margin-bottom:11px">WHY WE INDEX BY NAME · POR QUÉ INDEXAMOS POR NOMBRE</div>Pick the wrong numerical column and Python runs perfectly. You simply analyse the wrong measurement.</div>

That is why this notebook says `names.index("mean radius")` and not `X[:, 0]`
wherever the name is the thing actually meant.

The data is numerical measurements computed from digitized fine-needle
aspirate images. We use it to learn data operations, not to build a diagnostic
rule.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Si eliges por accidente otra columna numérica, Python sigue sin error: solo analizas la medición equivocada. Por eso indexamos por nombre, <code>names.index(&quot;mean radius&quot;)</code>, y no por <code>X[:, 0]</code>.<br><br>Son mediciones calculadas a partir de imágenes digitalizadas de aspiración con aguja fina. Las usamos para aprender operaciones con datos, no para construir una regla de diagnóstico.</div>

## 3.1 Indexing by name / Indexación por nombre

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#7c3aed,rgba(124,58,237,0))"></div>

Thirty measurement columns. Two ways to ask for one.

<div style="border-left:5px solid #7c3aed;background:rgba(124,58,237,.10);border-radius:0 8px 8px 0;padding:14px 18px;margin:1.8em 0;font:400 15px/1.75 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#7c3aed;margin-bottom:11px">SAY WHICH · DI CUÁL</div><div style="margin:.55em 0">❌ “Give me column 0.” / «Dame la columna 0.»</div><div style="margin:.55em 0">✅ “Give me the column called <code>mean radius</code>.” / «Dame la columna llamada <code>mean radius</code>.»</div></div>

The second says what you meant, and survives a change in column order.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>La segunda expresa la intención y sobrevive a un cambio en el orden de las columnas.</div>

## Exercise 1 — indexing by name / Ejercicio 1 — indexación por nombre

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#7c3aed,rgba(124,58,237,0))"></div>

Extract one real measurement for all 569 samples.

<div style="border-left:5px solid #7c3aed;background:rgba(124,58,237,.10);border-radius:0 8px 8px 0;padding:14px 18px;margin:1.8em 0;font:400 15px/1.75 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#7c3aed;margin-bottom:11px">PREDICT FIRST · PREDICE PRIMERO</div>If <span style="font:600 13px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;background:rgba(124,58,237,.14);border:1px solid rgba(124,58,237,.4);border-radius:999px;padding:4px 11px;white-space:nowrap">X.shape == (569, 30)</span> and you select one column, what shape comes back?</div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Extrae una medición real para las 569 muestras. Si <code>X.shape == (569, 30)</code> y seleccionas una columna, ¿qué forma debería salir?</div>

In [ ]:
# TODO 1 / TAREA 1
#
# EN:
# 1. Print X.shape and explain what each axis counts.
# 2. Find the position of "mean radius" with names.index(...).
# 3. Extract that column for all samples.
# 4. Print its shape.
#
# ES:
# 1. Imprime X.shape y explica qué cuenta cada eje.
# 2. Encuentra la posición de "mean radius" con names.index(...).
# 3. Extrae esa columna para todas las muestras.
# 4. Imprime su forma.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

print("X.shape / Forma de X:", X.shape)
print("EN: 569 samples × 30 measured features.")
print("ES: 569 muestras × 30 características medidas.")
print()

radius_idx = names.index("mean radius")
radius = X[:, radius_idx]

print("Column name / Nombre de columna:", names[radius_idx])
print("Column position / Posición de columna:", radius_idx)
print("radius.shape / Forma:", radius.shape)
print()
print("EN: selecting one column removes the feature axis from the result.")
print("ES: seleccionar una sola columna elimina el eje de características del resultado.")

### Feature explorer / Explorador de características

Pick any of the 30 feature names. You get its column position, its range, its
mean, and a histogram of the real measurements.

Indexing by name stops being abstract.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Elige cualquiera de las 30 características: posición de columna, rango, media y un histograma de las mediciones reales. La indexación por nombre deja de ser abstracta.</div>

In [ ]:
#@title 🔍 Feature explorer / Explorador de características — run me / ejecútame { display-mode: 'form' }

feature_dropdown = widgets.Dropdown(
    options=names,
    value="mean radius",
    description="Feature / Característica:",
    style={"description_width": "150px"},
)

def explore_feature(feature_name):
    i = names.index(feature_name)
    col = X[:, i]

    plt.close("all")
    fig, ax = plt.subplots(figsize=(6.5, 3.2))
    ax.hist(col, bins=30)
    ax.set_title(f"{feature_name} — real measurements / mediciones reales")
    ax.set_xlabel("value / valor")
    ax.set_ylabel("count / cantidad")
    plt.tight_layout()
    plt.show()

    print("Column / Columna:", i)
    print("Shape / Forma:", col.shape)
    print(f"Mean / Media: {col.mean():.3f}")
    print(f"Min / Mínimo: {col.min():.3f}")
    print(f"Max / Máximo: {col.max():.3f}")
    print("EN: one named feature has been selected for all 569 samples.")
    print("ES: se seleccionó una característica por nombre para las 569 muestras.")

feature_output = widgets.interactive_output(
    explore_feature,
    {"feature_name": feature_dropdown},
)

display(widgets.VBox([feature_dropdown, feature_output]))

<details>
<summary><strong>Why this solution works / Por qué funciona esta solución</strong></summary>

`X` has two axes: `(samples, features)`.

In `X[:, radius_idx]`:

- `:` keeps **all samples**;
- `radius_idx` keeps **one feature**.

One value per sample, so `(569,)`.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><code>X</code> tiene dos ejes: <code>(muestras, características)</code>. En <code>X[:, radius_idx]</code>, <code>:</code> conserva todas las muestras y <code>radius_idx</code> conserva una característica. Un valor por muestra: <code>(569,)</code>.</div>

</details>

## 3.2 Fancy indexing and Boolean masks / Fancy indexing y máscaras booleanas

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#7c3aed,rgba(124,58,237,0))"></div>

Two ideas, two different selection problems.

<div style="border-left:5px solid #7c3aed;background:rgba(124,58,237,.10);border-radius:0 8px 8px 0;padding:14px 18px;margin:1.8em 0;font:400 15px/1.75 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#7c3aed;margin-bottom:11px">WHICH ONE · CUÁL DE LAS DOS</div><div style="margin:.55em 0"><b>Fancy indexing</b> — you already know the positions. <code>X[[2, 10, 50], :]</code> is “rows 2, 10 and 50”.</div><div style="margin:.55em 0"><b>Boolean mask</b> — a condition defines the rows. <code>X[y == 0]</code> is “every row where <code>y == 0</code>”.</div></div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><b>Fancy indexing:</b> cuando ya conoces las posiciones. <b>Máscara booleana:</b> cuando una condición define las filas.</div>

## Exercise 2 — select observations / Ejercicio 2 — selecciona observaciones

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#7c3aed,rgba(124,58,237,0))"></div>

Two questions.

<div style="border-left:5px solid #7c3aed;background:rgba(124,58,237,.10);border-radius:0 8px 8px 0;padding:14px 18px;margin:1.8em 0;font:400 15px/1.75 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#7c3aed;margin-bottom:11px">ANSWER BOTH · RESPONDE LAS DOS</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#7c3aed;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">1</span>Which five samples have the largest <code>mean radius</code>?</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#7c3aed;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">2</span>How does <code>mean radius</code> differ on average, in this dataset, between the two target groups?</div></div>

The second answer is descriptive. It is not a one-feature diagnostic rule.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><b>1 ·</b> ¿qué cinco muestras tienen el <code>mean radius</code> más alto? <b>2 ·</b> ¿cómo difiere en promedio entre los dos grupos, dentro de este conjunto?<br><br>La segunda respuesta es descriptiva: no es una regla de diagnóstico de una sola característica.</div>

In [ ]:
# TODO 2 / TAREA 2
#
# EN:
# 1. Find the indices of the 5 largest values in `radius`.
# 2. Use fancy indexing to extract their full 30-feature profiles.
# 3. Verify that the result has shape (5, 30).
#
# ES:
# 1. Encuentra los índices de los 5 valores más grandes de `radius`.
# 2. Usa fancy indexing para extraer sus perfiles completos de 30 características.
# 3. Verifica que el resultado tenga forma (5, 30).
#
# TODO 3 / TAREA 3
#
# EN:
# Use Boolean masks to compute the mean radius for:
# - y == 0 (malignant)
# - y == 1 (benign)
#
# ES:
# Usa máscaras booleanas para calcular el radio medio para:
# - y == 0 (maligno)
# - y == 1 (benigno)

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

top5 = np.argsort(radius)[-5:]
profiles = X[top5, :]

print("Top-5 row indices / Índices de las 5 filas:", top5.tolist())
print("Profiles shape / Forma de perfiles:", profiles.shape)
print()

malignant_mask = (y == 0)
benign_mask = (y == 1)

malignant_mean = radius[malignant_mask].mean()
benign_mean = radius[benign_mask].mean()

print(f"Malignant mean radius / Radio medio maligno: {malignant_mean:.3f}")
print(f"Benign mean radius / Radio medio benigno: {benign_mean:.3f}")
print()
print("EN: in this dataset, the malignant group has a larger mean radius on average.")
print("ES: en este conjunto, el grupo maligno tiene un radio medio mayor en promedio.")
print("EN: this is a dataset description, not a diagnostic threshold.")
print("ES: esto describe el conjunto de datos; no es un umbral diagnóstico.")

### Mask explorer / Explorador de máscaras

Pick a group and a feature. The mask decides **which rows stay**; the feature
selector decides **which column you look at**.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Elige un grupo y una característica: la máscara decide <b>qué filas quedan</b> y el selector decide <b>qué columna miras</b>.</div>

In [ ]:
#@title 🎭 Mask explorer / Explorador de máscaras — run me / ejecútame { display-mode: 'form' }

group_toggle = widgets.ToggleButtons(
    options=[
        ("All / Todas", "all"),
        ("Malignant / Malignas", "malignant"),
        ("Benign / Benignas", "benign"),
    ],
    value="all",
    description="Rows / Filas:",
    style={"description_width": "100px"},
)

mask_feature = widgets.Dropdown(
    options=names,
    value="mean radius",
    description="Feature / Característica:",
    style={"description_width": "150px"},
)

def explore_mask(group, feature_name):
    i = names.index(feature_name)

    if group == "malignant":
        mask = (y == 0)
        group_en = "malignant"
        group_es = "maligno"
    elif group == "benign":
        mask = (y == 1)
        group_en = "benign"
        group_es = "benigno"
    else:
        mask = np.ones(len(y), dtype=bool)
        group_en = "all samples"
        group_es = "todas las muestras"

    selected = X[mask, i]

    plt.close("all")
    fig, ax = plt.subplots(figsize=(6.5, 3.2))
    ax.hist(selected, bins=30)
    ax.set_title(f"{feature_name} — {group_en} / {group_es}")
    ax.set_xlabel("value / valor")
    ax.set_ylabel("count / cantidad")
    plt.tight_layout()
    plt.show()

    print("True values in mask / Valores True en la máscara:", int(mask.sum()))
    print("Selected shape / Forma seleccionada:", selected.shape)
    print(f"Mean / Media: {selected.mean():.3f}")
    print("EN: True means 'keep this row'.")
    print("ES: True significa 'conservar esta fila'.")

mask_output = widgets.interactive_output(
    explore_mask,
    {"group": group_toggle, "feature_name": mask_feature},
)

display(widgets.VBox([group_toggle, mask_feature, mask_output]))

### Compare all 30 features / Compara las 30 características

`mean radius` is one measurement out of thirty. Walk the list and compare the
two group distributions. Some separate visibly. Most do not.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><code>mean radius</code> es una de treinta. Recorre la lista y compara las distribuciones de ambos grupos: algunas se separan a la vista, la mayoría no.</div>

In [ ]:
#@title 📊 Feature comparison / Comparación de características — run me / ejecútame { display-mode: 'form' }

comparison_feature = widgets.Dropdown(
    options=names,
    value="mean radius",
    description="Feature / Característica:",
    style={"description_width": "150px"},
)

def compare_groups(feature_name):
    i = names.index(feature_name)
    col = X[:, i]

    plt.close("all")
    fig, ax = plt.subplots(figsize=(6.8, 3.4))
    ax.hist(col[y == 0], bins=30, alpha=0.6, label="malignant / maligno")
    ax.hist(col[y == 1], bins=30, alpha=0.6, label="benign / benigno")
    ax.set_title(feature_name)
    ax.set_xlabel("value / valor")
    ax.set_ylabel("count / cantidad")
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f"Malignant mean / Media maligna: {col[y == 0].mean():.3f}")
    print(f"Benign mean / Media benigna: {col[y == 1].mean():.3f}")
    print("EN: overlap matters; one feature alone is not a diagnosis.")
    print("ES: la superposición importa; una sola característica no constituye un diagnóstico.")

comparison_output = widgets.interactive_output(
    compare_groups,
    {"feature_name": comparison_feature},
)

display(widgets.VBox([comparison_feature, comparison_output]))

## 3.3 Broadcasting on real images / Broadcasting sobre imágenes reales

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#7c3aed,rgba(124,58,237,0))"></div>

From selecting data to transforming it.

Each digit is an `8 × 8` image. Flattened, each becomes a row of 64 pixel
features: <span style="font:600 13px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;background:rgba(124,58,237,.14);border:1px solid rgba(124,58,237,.4);border-radius:999px;padding:4px 11px;white-space:nowrap">(1797, 8, 8) → (1797, 64)</span>.

We want to standardize all 64 pixel columns.

<div style="border-left:5px solid #7c3aed;background:rgba(124,58,237,.10);border-radius:0 8px 8px 0;padding:14px 18px;margin:1.8em 0;font:400 15px/1.75 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#7c3aed;margin-bottom:11px">THE ANALOGY · LA ANALOGÍA</div>One mean per column gives <code>mean.shape = (64,)</code>. NumPy subtracts those 64 numbers from <b>every one</b> of the 1,797 rows, as if the row had been copied 1,797 times — without copying it.<div style='margin-top:10px'>That is broadcasting.</div></div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Una media por columna da <code>(64,)</code>. NumPy resta esos 64 números a <b>cada una</b> de las 1.797 filas, como si el vector se hubiera copiado 1.797 veces — sin copiarlo. Eso es broadcasting.</div>

In [ ]:
D = images.reshape(len(images), -1)

print("Original images / Imágenes originales:", images.shape)
print("Flattened matrix / Matriz aplanada:", D.shape)
print("EN: each row is one image; each column is one pixel position.")
print("ES: cada fila es una imagen; cada columna es una posición de píxel.")

### Broadcasting shape explorer / Explorador de formas de broadcasting

One question matters: **can a `(64,)` vector line up with the last axis of a
`(1797, 64)` matrix?**

Step through the subtraction and the division.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>La pregunta que importa: ¿puede un vector <code>(64,)</code> alinearse con el último eje de una matriz <code>(1797, 64)</code>? Recorre la resta y la división.</div>

In [ ]:
#@title 🔍 Broadcasting explorer / Explorador de broadcasting — run me / ejecútame { display-mode: 'form' }

broadcast_step = widgets.ToggleButtons(
    options=[
        ("Original / Original", "original"),
        ("Subtract mean / Restar media", "center"),
        ("Divide by std / Dividir por std", "standardize"),
    ],
    value="original",
    description="Step / Paso:",
    style={"description_width": "100px"},
)

mean = D.mean(axis=0)
std = D.std(axis=0)
zero_variance = std == 0
safe_std = np.where(std == 0, 1.0, std)

def show_broadcast_step(step):
    if step == "original":
        arr = D
        en = "raw pixel values"
        es = "valores de píxel originales"
    elif step == "center":
        arr = D - mean
        en = "the 64 means are subtracted from every image row"
        es = "las 64 medias se restan de cada fila de imagen"
    else:
        arr = (D - mean) / safe_std
        en = "each centered pixel column is divided by its standard deviation"
        es = "cada columna de píxel centrada se divide por su desviación estándar"

    print("Input matrix / Matriz:", D.shape)
    print("mean.shape / forma de media:", mean.shape)
    print("std.shape / forma de std:", std.shape)
    print("Output / Salida:", arr.shape)
    print("EN:", en)
    print("ES:", es)

broadcast_output = widgets.interactive_output(
    show_broadcast_step,
    {"step": broadcast_step},
)

display(widgets.VBox([broadcast_step, broadcast_output]))

## Exercise 3 — standardize, then find the trap / Ejercicio 3 — estandariza y encuentra la trampa

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#7c3aed,rgba(124,58,237,0))"></div>

The usual formula is `Z = (D - mean) / std`.

Real data has a surprise waiting in it.

<div style="border-left:5px solid #d97706;background:rgba(217,119,6,.10);border-radius:0 8px 8px 0;padding:14px 18px;margin:1.8em 0;font:400 15px/1.75 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#d97706;margin-bottom:11px">PREDICT FIRST · PREDICE PRIMERO</div>One pixel column has <code>std = 0</code>. What happens when the formula divides by it?</div>

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>La fórmula habitual es <code>Z = (D - mean) / std</code>, pero los datos reales guardan una sorpresa: ¿qué ocurre si una columna de píxeles tiene <code>std = 0</code> y dividimos por ese valor?</div>

In [ ]:
# TODO 4 / TAREA 4
#
# EN:
# 1. Compute the mean and std of each of the 64 pixel columns.
# 2. Print mean.shape and std.shape.
# 3. Try Z_bad = (D - mean) / std.
# 4. Check whether Z_bad contains NaN.
#
# ES:
# 1. Calcula la media y std de cada una de las 64 columnas de píxeles.
# 2. Imprime mean.shape y std.shape.
# 3. Prueba Z_bad = (D - mean) / std.
# 4. Comprueba si Z_bad contiene NaN.
#
# TODO 5 / TAREA 5
#
# EN:
# 1. Count how many pixels have std == 0.
# 2. Explain why a real handwritten-digit dataset could contain such pixels.
# 3. Replace zero denominators safely and verify that NaN disappears.
#
# ES:
# 1. Cuenta cuántos píxeles tienen std == 0.
# 2. Explica por qué un conjunto real de dígitos manuscritos puede contener esos píxeles.
# 3. Sustituye de forma segura los denominadores cero y verifica que desaparezcan los NaN.

In [ ]:
#@title Solution / Solución — try it yourself first / inténtalo primero { display-mode: 'form' }

mean = D.mean(axis=0)
std = D.std(axis=0)

print("mean.shape / Forma de mean:", mean.shape)
print("std.shape / Forma de std:", std.shape)

with np.errstate(divide="ignore", invalid="ignore"):
    Z_bad = (D - mean) / std

print("NaN before fix / NaN antes de corregir:", bool(np.isnan(Z_bad).any()))

zero_variance = (std == 0)
zero_count = int(zero_variance.sum())

print("Zero-variance pixels / Píxeles de varianza cero:", zero_count)

safe_std = np.where(zero_variance, 1.0, std)
Z = (D - mean) / safe_std

print("NaN after fix / NaN después de corregir:", bool(np.isnan(Z).any()))
print()
print("EN: zero-variance pixels never change across the 1,797 images.")
print("ES: los píxeles de varianza cero nunca cambian entre las 1.797 imágenes.")
print("EN: dividing by zero created invalid values; the safe denominator prevents that.")
print("ES: dividir entre cero creó valores inválidos; el denominador seguro lo evita.")

fig, ax = plt.subplots(figsize=(3.6, 3.6))
ax.imshow(D.mean(axis=0).reshape(8, 8), cmap="gray")

zero_rows, zero_cols = np.where(zero_variance.reshape(8, 8))
ax.scatter(zero_cols, zero_rows, s=220, marker="s", facecolors="none")
ax.set_title("Zero-variance pixels / Píxeles de varianza cero")
ax.axis("off")

plt.tight_layout()
plt.show()

### Zero-variance explorer / Explorador de varianza cero

Pick a pixel position from `0` to `63`. You get its row and column in the
`8 × 8` image, its mean, its standard deviation, whether it is constant, and
every value it takes across all 1,797 images.

Try a highlighted zero-variance pixel, then one near the centre of the digit.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>Elige una posición de píxel entre <code>0</code> y <code>63</code>: fila y columna, media, desviación estándar, si es constante, y todos sus valores en las 1.797 imágenes. Prueba un píxel de varianza cero y después uno del centro del dígito.</div>

In [ ]:
#@title 🔍 Zero-variance explorer / Explorador de varianza cero — run me / ejecútame { display-mode: 'form' }

zero_indices = np.flatnonzero(zero_variance)

pixel_slider = widgets.IntSlider(
    value=int(zero_indices[0]) if len(zero_indices) else 0,
    min=0,
    max=63,
    step=1,
    description="Pixel / Píxel:",
    continuous_update=False,
    style={"description_width": "100px"},
)

def explore_pixel(pixel):
    values = D[:, pixel]
    row, col = divmod(pixel, 8)

    plt.close("all")
    fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.2))

    axes[0].imshow(D.mean(axis=0).reshape(8, 8), cmap="gray")
    axes[0].scatter([col], [row], s=180, marker="s", facecolors="none")
    axes[0].set_title(f"Pixel {pixel} → ({row}, {col})")
    axes[0].axis("off")

    axes[1].hist(values, bins=20)
    axes[1].set_title("Values across images / Valores entre imágenes")
    axes[1].set_xlabel("pixel value / valor del píxel")
    axes[1].set_ylabel("count / cantidad")

    plt.tight_layout()
    plt.show()

    print("Mean / Media:", float(values.mean()))
    print("Std / Desviación estándar:", float(values.std()))
    print("Zero variance / Varianza cero:", bool(values.std() == 0))
    if values.std() == 0:
        print("EN: every image has exactly the same value at this pixel.")
        print("ES: todas las imágenes tienen exactamente el mismo valor en este píxel.")
    else:
        print("EN: this pixel changes across images and therefore carries variation.")
        print("ES: este píxel cambia entre imágenes y por eso contiene variación.")

pixel_output = widgets.interactive_output(
    explore_pixel,
    {"pixel": pixel_slider},
)

display(widgets.VBox([pixel_slider, pixel_output]))

<details>
<summary><strong>Why did NaN appear? / ¿Por qué apareció NaN?</strong></summary>

Standardization divides by the standard deviation. A pixel that never changes
across every image has `std = 0`, so the formula asks NumPy to divide by zero,
and `NaN` is what comes back.

**The code revealed a property of the data.** A zero-variance feature is not
automatically a bug in your program.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div>La estandarización divide por la desviación estándar. Un píxel que nunca cambia tiene <code>std = 0</code>, la fórmula divide entre cero y sale <code>NaN</code>.<br><br><b>El código reveló una propiedad de los datos.</b> Una característica de varianza cero no es automáticamente un error de programación.</div>

</details>

## What just happened / Qué acaba de pasar

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#7c3aed,rgba(124,58,237,0))"></div>

Three ways to select, one way to transform, all on real data.

<div style="border-left:5px solid #7c3aed;background:rgba(124,58,237,.10);border-radius:0 8px 8px 0;padding:14px 18px;margin:1.8em 0;font:400 15px/1.75 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 11px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;color:#7c3aed;margin-bottom:11px">FOUR IDEAS · CUATRO IDEAS</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#7c3aed;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">1</span><b>Index by meaning.</b> A feature name says what you meant; a column number does not.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#7c3aed;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">2</span><b>Fancy indexing chooses positions.</b> Use it when you know which rows.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#7c3aed;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">3</span><b>Boolean masks choose by condition.</b> <code>True</code> means “keep this one”.</div><div style="margin:.55em 0"><span style="display:inline-block;width:22px;height:22px;border-radius:50%;background:#7c3aed;color:#fff;text-align:center;font:700 12px/22px ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-right:9px">4</span><b>Broadcasting exposes real-data traps.</b> Standardization failed at exactly the columns whose standard deviation was zero.</div></div>

Two findings, from these two datasets:

- `mean radius` is larger on average in the malignant group than in the benign
  group.
- Exactly **three pixel positions** are constant across all 1,797 digit images.

Both are properties of *these* datasets. Neither is a general rule.

<div style="border-left:4px solid rgba(130,130,150,.5);background:rgba(130,130,150,.09);border-radius:0 8px 8px 0;padding:12px 16px;margin:1.2em 0 1.8em;font:400 14.5px/1.7 ui-sans-serif,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif"><div style="font:700 10.5px/1 ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;letter-spacing:.18em;opacity:.62;margin-bottom:10px">🇪🇸 ESPAÑOL</div><b>1 ·</b> indexa por significado; <b>2 ·</b> fancy indexing elige posiciones; <b>3 ·</b> las máscaras eligen por condición; <b>4 ·</b> el broadcasting destapa trampas de los datos reales.<br><br>Dos hallazgos <i>de estos conjuntos</i>: <code>mean radius</code> es mayor en promedio en el grupo maligno, y exactamente <b>tres posiciones de píxel</b> son constantes en las 1.797 imágenes. No son reglas generales.</div>

<div style="height:3px;border-radius:2px;margin:1.4em 0 1.6em;background:linear-gradient(90deg,#7c3aed,rgba(124,58,237,0))"></div>

## Done with this section / Fin de esta sección

Next / Siguiente: **04 · Reshape and transpose real images / Reshape y transposición de imágenes reales** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/04-reshape-and-transpose.ipynb).

[← Workshop site / Sitio del taller](https://project-delphi.github.io/tensors-workshop/) · [All notebooks / Todos los notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook / Manual](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)